In [22]:
! pip install pymysql
! pip install neo4j

  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 3.2 MB/s eta 0:00:00a 0:00:01m
Using cached pytz-2026.1.post1-py2.py3-none-any.whl (510 kB)


In [ ]:
import pymysql
from neo4j import GraphDatabase, exceptions

In [5]:
# host = input("Enter your host name: ")
# username = input("Enter your user name: ")
# password = input("Enter your password: ")
# db = input("Enter the name of the database: ")

In [ ]:
uri = "neo4j://localhoast:7687"
neo4jDriver = GraphDatabase.driver(uri, auth=("username", "password"))

In [ ]:
conn = pymysql.connect(
    host="localhost", 
    user="root", 
    password="xxx", 
    database="appdbproj", 
    cursorclass=pymysql.cursors.DictCursor, 
    port=3306
    )

In [ ]:
def print_menu():
    print("""Conference Management
--------------------------

MENU
====
1 - View Speakers & Sessions
2 - View Attendees by Company
3 - Add New Attendee
4 - View Connected Attendees
5 - Add Attendee Connection
6 - View Rooms
x - Exit application""")
    choice = input("Choice: ")
    return choice


def sql_queries():
    
    while True:
        choice = print_menu()

        if choice == "1":
            speaker_name = input("Enter speaker name: ")
            query = """SELECT a.attendeeName AS Name, s.sessionTitle AS session, rm.roomName AS room 
                    FROM attendee AS a 
                    RIGHT JOIN registration AS r ON a.attendeeID = r.attendeeID 
                    RIGHT JOIN session AS s ON r.sessionID = s.sessionID 
                    RIGHT JOIN room AS rm ON s.roomID = rm.roomID
                    WHERE a.attendeeName LIKE %s"""
            
            with conn.cursor() as cursor:
                cursor.execute(query, (f"%{speaker_name}%",))
                subjects = cursor.fetchall()
                if not subjects:
                    print("No speaker found of that name")
                else:
                    print("""
-----------------------------------------------""")
                    for s in subjects:
                        print(f"{s['Name']}  |  {s['session']}  |  {s['room']}")
                    print("""-----------------------------------------------
                          """)
        
        if choice == "2":

            with conn.cursor() as cursor:
                
                while True:
                    
                    companyID = input("Enter company ID: ")

                    cursor.execute("SELECT companyName FROM company WHERE companyID = %s", (companyID,))
                    company = cursor.fetchone()
                    
                    
                    if not company:
                        print(f"Company with ID {companyID} does not exist")
                        continue

                    cursor.execute("SELECT * FROM attendee WHERE attendeeCompanyID = %s", (companyID,))
                    attendees = cursor.fetchall()

                    if not attendees:
                        print(f"No attendees found for {company["companyName"]}")
                        continue
                    
                    query = """SELECT a.attendeeName AS Name, a.attendeeDOB AS DOB, s.sessionTitle AS sessionTitle, 
                    s.speakerName AS sessionSpeaker, rm.roomName AS roomName
                    FROM attendee AS a 
                    RIGHT JOIN registration AS r ON a.attendeeID = r.attendeeID 
                    RIGHT JOIN session AS s ON r.sessionID = s.sessionID 
                    RIGHT JOIN room AS rm ON s.roomID = rm.roomID
                    RIGHT JOIN company AS c ON a.attendeeCompanyID = c.companyID
                    WHERE c.companyID = %s"""

                    cursor.execute(query, (companyID,))
                    subjects = cursor.fetchall()
                    print()
                    print("-----------------------------------------------")
                    print(company["companyName"])
                    for s in subjects:
                        print(f"{s['Name']}  |  {s['DOB']}  |  {s['sessionTitle']}  |  {s['sessionSpeaker']}  |  {s['roomName']}")
                    print("-----------------------------------------------")
                    print()
                    break

        if choice == "3":
            print("""
Add New Attendee
----------------""")
            attendeeID = input("Enter attendee ID: ")
            attendee_name = input("Enter attendee's first name and surname: ")
            attendeeDOB = input("Enter the date of birth of attendee (YYYY-MM-DD): ")
            attendee_gender = input("Enter the gender of the attendee: ")
            attendee_company = input("Enter the ID of the attendee's company: ")

            with conn.cursor() as cursor:

                cursor.execute("SELECT attendeeID FROM attendee WHERE attendeeID = %s", (attendeeID,))
                attendee = cursor.fetchone()

                if attendee:
                    print(f"*** ERROR *** Attendee ID: {attendeeID} already exist")
                
                gender_correct = attendee_gender != "Male" or attendee_gender != "Female"

                if gender_correct:
                    print("*** ERROR *** Gender must be Male/Female")
                
                cursor.execute("SELECT companyName FROM company WHERE companyID = %s", (attendee_company,))
                company = cursor.fetchone()
                    
                if not company:
                    print(f"*** ERROR *** Company ID: {attendee_company} does not exist")
                    
                

                if not attendee and company and gender_correct:

                    try:
                    
                        query = "INSERT INTO attendee VALUES (%s, %s, %s, %s, %s)"


                        cursor.execute(query, (attendeeID, attendee_name, attendeeDOB, attendee_gender, attendee_company))
                        conn.commit()  
                        print(f"Attendee {attendee_name} successfully added.")

                        cursor.execute("SELECT * FROM attendee WHERE attendeeID = %s", (attendeeID,))
                        new_attendee = cursor.fetchone()


                        print("-----------------------------------------------")
                        print(f"ID: {new_attendee['attendeeID']}  |  Name: {new_attendee['attendeeName']}  |  DOB: {new_attendee['attendeeDOB']}  |  Gender: {new_attendee['attendeeGender']}  |  Company ID: {new_attendee['attendeeCompanyID']}")
                        print("-----------------------------------------------")
                        print()

                    except pymysql.Error as e:
                        print(f"*** Error *** {e}")
                        conn.rollback()


        if choice == "4":
            

        if choice == "x":
            print("Exiting...")
            break

In [20]:
sql_queries()

Conference Management
--------------------------

MENU
====
1 - View Speakers & Sessions
2 - View Attendees by Company
3 - Add New Attendee
4 - View Connected Attendees
5 - Add Attendee Connection
6 - View Rooms
x - Exit application

Add New Attendee
----------------
*** ERROR *** Gender must be Male/Female
*** Error *** (1292, "Incorrect date value: 'd' for column 'attendeeDOB' at row 1")
Conference Management
--------------------------

MENU
====
1 - View Speakers & Sessions
2 - View Attendees by Company
3 - Add New Attendee
4 - View Connected Attendees
5 - Add Attendee Connection
6 - View Rooms
x - Exit application
Exiting...
